# Kaggle SPICE Self-Play Training

This notebook implements the **SPICE (Self-Play In Corpus Environments)** framework for OnCallEnv Red Shift. It trains a single model to act as both an **LLM Attacker** (generating hard scenarios) and an **LLM Defender** (solving them) using **DrGRPO**.

## 1. GPU Check

Expected: Two Tesla T4 GPUs.

In [1]:
!nvidia-smi

Sun Apr 26 01:17:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|


|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+
|   1  Tesla T4                       Off |   00000000:00:05.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        

## 2. Bootstrap Repository

Clones or updates the repository in `/kaggle/working`.

In [18]:
import os, shutil, subprocess, time
from pathlib import Path

REPO_URL = "https://github.com/srimanreddy4/MetaHackathon-R2"
BRANCH = "spicy-attacker"
WORKDIR = Path("/kaggle/working/MetaHackathon-R2")

os.chdir("/kaggle/working")
if (WORKDIR / ".git").exists():
    os.chdir(WORKDIR)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(WORKDIR)], check=True)
    os.chdir(WORKDIR)

print("cwd:", os.getcwd())
subprocess.run(["git", "log", "--oneline", "-5"], check=True)

From https://github.com/srimanreddy4/MetaHackathon-R2
 * branch            spicy-attacker -> FETCH_HEAD
   f6752c5..5438909  spicy-attacker -> origin/spicy-attacker
Already on 'spicy-attacker'


Your branch is behind 'origin/spicy-attacker' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
Updating f6752c5..5438909
Fast-forward
 notebooks/05_kaggle_spice_selfplay.ipynb | 172 ++++++++++++++++++++++++-------
 scripts/llm_attacker.py                  |   4 +-
 scripts/train_spice_selfplay.py          |   6 +-
 3 files changed, 141 insertions(+), 41 deletions(-)
cwd: /kaggle/working/MetaHackathon-R2
5438909 lowered defender passing thresh
f6752c5 properly ignore large files and commit code changes
303b91d split attacker and defender dataset
a1fcebf fix?
3d81be1 changed reward to binary, increase groupsize


From https://github.com/srimanreddy4/MetaHackathon-R2
 * branch            spicy-attacker -> FETCH_HEAD


CompletedProcess(args=['git', 'log', '--oneline', '-5'], returncode=0)

In [5]:
!python scripts/debug_spice_iter.py


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-04-26 01:20:06.324267: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777166406.537779     169 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777166406.605042     169 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777166407.140375     169 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777166407.140454     169 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:17

## 3. Install Dependencies

Installs standard requirements and LLM training stack (Unsloth, TRL).

In [4]:
!python -m pip install -U pip setuptools wheel
!python -m pip install -r requirements.txt
!python -m pip install -r requirements-llm.txt
!python -m pip install pytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.0 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.46.3
    Uninstalling wheel-0.46.3:
      Successfully uninstalled wheel-0.46.3
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 19.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [openenv-core] [fastmcp]]ydantic]
INFO: pip is looking at multiple versions of unsloth to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of unsloth to det

## 4. Global Environment Setup

In [6]:
%cd /kaggle/working/MetaHackathon-R2
%env PYTHONPATH=src:scripts
import sys
sys.path.append("src")
sys.path.append("scripts")

/kaggle/working
env: PYTHONPATH=src:scripts


In [7]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import os

# Fetch the token from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Set it as an environment variable (Transformers uses this automatically)
os.environ["HF_TOKEN"] = hf_token

# And/or explicitly login
login(token=hf_token)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 5. Verify Attacker Logic

Run the unit tests to ensure the discrete action parser and variance rewards are correct on this kernel.

In [14]:
!python -m pytest tests/test_llm_attacker.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working/MetaHackathon-R2
configfile: pyproject.toml
plugins: anyio-4.12.1, langsmith-0.7.6, typeguard-4.5.1
collected 25 items                                                             

tests/test_llm_attacker.py::TestParseAttackerActions::test_single_valid_action PASSED [  4%]
tests/test_llm_attacker.py::TestParseAttackerActions::test_multiple_valid_actions PASSED [  8%]
tests/test_llm_attacker.py::TestParseAttackerActions::test_invalid_field_name_ignored PASSED [ 12%]
tests/test_llm_attacker.py::TestParseAttackerActions::test_invalid_value_ignored PASSED [ 16%]
tests/test_llm_attacker.py::TestParseAttackerActions::test_no_actions_returns_invalid PASSED [ 20%]
tests/test_llm_attacker.py::TestParseAttackerActions::test_empty_actions_block PASSED [ 24%]
tests/test_llm_attacker.py::Tes

## 6. SPICE Self-Play Smoke Run

Tests the full interaction loop between Attacker and Defender with minimal steps.

In [7]:
!python scripts/train_spice_selfplay.py \
    --selfplay-iterations 2 \
    --group-size 2 \
    --batch-size 2 \
    --max-steps 5 \
    --out-dir "training_results/spice_smoke" \
    --report-to "none" \
    --verbose

Loaded 120 parent scenarios
/kaggle/working/MetaHackathon-R2/scripts/train_spice_selfplay.py:578: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, PatchFastRL
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-04-25 21:52:17.909278: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777153937.933417     878 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777153937.941438     878 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register facto

## 7. Main SPICE Self-Play Training

Runs the primary co-evolution loop. Adjust `--report-to` to `wandb` if you have it configured.

In [ ]:
!pip uninstall -y vllm


Found existing installation: vllm 0.19.1
Uninstalling vllm-0.19.1:
  Successfully uninstalled vllm-0.19.1


In [ ]:
!python scripts/train_spice_selfplay.py \
    --model-name "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit" \
    --selfplay-iterations 5 \
    --group-size 4 \
    --selfplay-group-size 4 \
    --batch-size 2 \
    --max-steps 200 \
    --per-device-train_batch_size 1 \
    --gradient-accumulation-steps 8 \
    --lr 5e-6 \
    --report-to "tensorboard" \
    --out-dir "training_results/spice_selfplay_v2" \
    --save-steps 25 \
    --load-dataset "training_results/spice_selfplay_v2/selfplay_dataset.jsonl" \
    --resume-from-checkpoint "training_results/spice_selfplay_v2/checkpoint-50" \
    --push-to-hub \
    --hub-model-id "NeerjaK/spice-qwen-1.5b-grpo"


Loaded 120 parent scenarios
/kaggle/working/MetaHackathon-R2/scripts/train_spice_selfplay.py:580: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, PatchFastRL
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-04-26 03:20:05.127542: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777173605.151782    1220 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777173605.159715    1220 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register facto

In [ ]:
!python scripts/train_spice_selfplay.py \
    --model-name "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit" \
    --selfplay-iterations 5 \
    --group-size 4 \
    --selfplay-group-size 4 \
    --batch-size 2 \
    --max-steps 200 \
    --per-device-train_batch_size 1 \
    --gradient-accumulation-steps 8 \
    --lr 5e-6 \
    --report-to "tensorboard" \
    --out-dir "training_results/spice_selfplay_v2" \
    --save-steps 25 \
    --load-dataset "training_results/spice_selfplay_v2/selfplay_dataset.jsonl" \
    --resume-from-checkpoint "NeerjaK/spice-qwen-1.5b-grpo" \
    --push-to-hub \
    --hub-model-id "NeerjaK/spice-qwen-1.5b-grpo"


## 8. Summary & Results

In [ ]:
!cat training_results/spice_selfplay_main/summary.json

cat: training_results/spice_selfplay_main/summary.json: No such file or directory


In [ ]:
import shutil
# Zip the entire results folder
shutil.make_archive('training_results', 'zip', 'training_results/spice_selfplay_v2')
print("Zipping complete: spice_training_results.zip")


Zipping complete: spice_training_results.zip


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def plot_spice_results(run_dir_path):
    run_dir = Path(run_dir_path)
    
    # --- Part 1: Iteration-wise Co-evolution (from Generations) ---
    # Since summary.json is only saved at the end, we look for generation files
    a_gen_path = run_dir / "attacker_generations.json"
    d_gen_path = run_dir / "defender_generations.json"
    
    if a_gen_path.exists() and d_gen_path.exists():
        with open(a_gen_path, "r") as f: a_data = json.load(f)
        with open(d_gen_path, "r") as f: d_data = json.load(f)
        
        # Plotting Attacker vs Defender co-evolution
        # Note: If iterations aren't explicitly labeled, we can still see the trend
        a_rewards = [row.get("reward", 0) for row in a_data]
        d_rewards = [row.get("reward", row.get("raw_reward", 0)) for row in d_data]
        
        plt.figure(figsize=(10, 4))
        plt.plot(a_rewards, label="Attacker Reward", alpha=0.5, color="red")
        plt.plot(d_rewards, label="Defender Reward", alpha=0.5, color="blue")
        plt.title("Self-Play Generation Rewards (Raw Samples)")
        plt.xlabel("Sample Index")
        plt.ylabel("Reward")
        plt.legend()
        plt.grid(True, alpha=0.2)
        plt.show()

    # --- Part 2: GRPO Training (from trainer_state.json) ---
    state_path = run_dir / "trainer_state.json"
    if not state_path.exists():
        checkpoints = sorted(run_dir.glob("checkpoint-*"), key=lambda x: int(x.name.split("-")[1]))
        if checkpoints:
            state_path = checkpoints[-1] / "trainer_state.json"
            
    if state_path.exists():
        with open(state_path, "r") as f:
            state = json.load(f)
        
        history = state.get("log_history", [])
        metrics = []
        for row in history:
            if "loss" in row or "reward" in row:
                metrics.append({
                    "step": row.get("step"),
                    "loss": row.get("loss"),
                    "reward": row.get("reward"),
                    "kl": row.get("kl")
                })
        
        if metrics:
            df = pd.DataFrame(metrics).dropna(subset=["step"])
            df.to_csv(run_dir / "spice_training_metrics.csv", index=False)
            
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
            
            # GRPO Reward
            ax1.plot(df["step"], df["reward"], color="green", label="GRPO Aggregate Reward")
            ax1.set_title("GRPO Training Reward")
            ax1.set_xlabel("Step")
            ax1.grid(True, alpha=0.3)
            
            # GRPO Loss
            ax2.plot(df["step"], df["loss"], color="orange", label="GRPO Loss")
            ax2.set_title("GRPO Training Loss")
            ax2.set_xlabel("Step")
            ax2.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
            print(f"GRPO metrics stored in {run_dir / 'spice_training_metrics.csv'}")
    else:
        print("Training state not found yet. GRPO phase might not have started.")

# Run for your SPICE directory
plot_spice_results("training_results/spice_selfplay_v2")


## 8. Export Results

Archive the training results for download from Kaggle.